In [ ]:
# import os
# os.chdir("../VITAL")

# from openai import AzureOpenAI

# api_key = ""
# azure_endpoint = ""
# model_name = "o4-mini"

# # gets the API Key from environment variable AZURE_OPENAI_API_KEY
# client = AzureOpenAI(
#     api_version="2025-01-01-preview",
#     api_key=api_key,
#     azure_endpoint=azure_endpoint,
# )

# A = 20
# import json, random
# def get_aug_dict(context, str_cols, df_train):
#     text_dict = {}
#     for str_col in str_cols:
#         text_dict[str_col] = {}
#         for unique_str in list(df_train[str_col].astype(str).unique()):
#             # augment this unique_str for A times, using openai
#             prompt = f"With simple changes, generate {A} different sentences that has the same meaning as this sentence related to {context}: '{unique_str}'. Split by '|' and return the list of sentences."
#             completion = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": prompt,
#                     },
#                 ],
#             )
#             response = json.loads(completion.model_dump_json(indent=2))['choices'][0]['message']['content']
#             aug_strs = response.split('|')
#             aug_strs = [s.strip() + ('' if s.strip().endswith('.') else '.') for s in aug_strs if s.strip()] # # Ensure each string ends with a period
#             text_dict[str_col][unique_str] = random.sample(aug_strs, k=A) # shuffle
#     return text_dict


In [ ]:
import os, json, random
from openai import OpenAI           # pip install --upgrade openai
from key import openai_key
client = OpenAI(api_key=openai_key)

os.chdir("../VITAL")

MODEL_NAME = "gpt-4o"          # official public model name
A = 50                              # number of augmentations per input string


def get_aug_dict(context: str, str_cols, df_train):
    """
    Generate A paraphrases for every distinct string in the given columns of df_train.

    Returns
    -------
    dict[str, dict[str, list[str]]]
        {column_name: {original_string: [augmented_str1, …, augmented_strA]}}
    """
    text_dict = {}

    for col in str_cols:
        text_dict[col] = {}
        unique_vals = df_train[col].dropna().unique() 
        for original in map(str, unique_vals):
            if original.strip() == 'No events.':
                original = 'No Bradycardia events.'
            prompt = (
                f"Generate {A} different sentences that have the " #With simple changes, 
                f"same meaning as this sentence related to {context}: '{original}'. "
                "Only return sentences, do not number the sentences, split the sentences with the character '|' only."
            )

            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,         # add diversity; tweak as needed
                max_tokens=2048           # plenty for 20 short sentences
            )

            reply = completion.choices[0].message.content.strip()
            print(reply)
            aug_strs = [
                s if s.endswith('.') else s + '.'
                for s in (seg.strip() for seg in reply.split('|')) if s
            ]

            if len(aug_strs) < A:             # safety fallback
                aug_strs *= (A // len(aug_strs) + 1)

            text_dict[col][original] = random.sample(aug_strs, k=A)

    return text_dict


# Air quality

In [ ]:
overwrite = False
dataset_name = 'air'
attr_suffix = ''
suffix = '' 
with open('run/settings.py', 'r') as file:
    exec(file.read())
with open('run/data.py', 'r') as file:
    exec(file.read())

In [ ]:
text_dict = get_aug_dict(
    context  = 'air quality', 
    str_cols = ['city_str', 'year_str', 'season_str'],
    df_train = df_train
)
with open("../../data/air_quality/aug_text.json", "w") as f:
    json.dump(text_dict, f, indent=2, ensure_ascii=False)

# Synthetic

In [ ]:
overwrite = False
dataset_name = 'syn'
attr_suffix = ''
suffix = '' 
with open('run/settings.py', 'r') as file:
    exec(file.read())
with open('run/data.py', 'r') as file:
    exec(file.read())

In [ ]:
text_dict = get_aug_dict(
    context  = 'time series', 
    str_cols = ['segment1', 'segment2', 'segment3', 'segment4'],
    df_train = df_train
)
with open("../../data/synthetic/aug_text.json", "w") as f:
    json.dump(text_dict, f, indent=2, ensure_ascii=False)

# NICU

In [ ]:
overwrite = False
dataset_name = 'nicu'
attr_suffix = ''
suffix = '' 
with open('run/settings.py', 'r') as file:
    exec(file.read())
with open('run/data.py', 'r') as file:
    exec(file.read())

In [ ]:
text_dict = get_aug_dict(
    context  = 'NICU heart rate', 
    str_cols = ['description_succ_inc', 'description_histogram', 'description_ts_event', 'description_ts_event_binary'],
    df_train = df_train
)
with open("../../data/nicu/aug_text.json", "w") as f:
    json.dump(text_dict, f, indent=2, ensure_ascii=False)

# Synthetic (open_gen)

In [ ]:
df_ls = []
for attr_id in [1, 3, 4]:
    loo_text = ""

    overwrite = False
    dataset_name = 'syn_gt'
    attr_suffix = ''
    suffix = ''

    with open('run/settings.py', 'r') as file:
        exec(file.read())
    model_name = ''.join([dataset_name, attr_suffix, suffix]) 
    exec(open('run/open_gen/configs/synthetic_gen.py', 'r').read())
    exec(open('run/open_gen/prepare_datasets/synthetic_gen.py', 'r').read())
    df_ls.append(df_left[['segment' + str(attr_id)]].reset_index(drop=True))

df = pd.concat(df_ls, axis = 1,ignore_index=True) # by row
df.columns = ['segment' + str(i) for i in [1, 3, 4]]


In [ ]:
text_dict = get_aug_dict(
    context  = 'time series', 
    str_cols = ['segment1', 'segment3', 'segment4'],
    df_train = df
)
with open("../../data/synthetic/aug_text_gen.json", "w") as f:
    json.dump(text_dict, f, indent=2, ensure_ascii=False)